In [3]:
import numpy as np 
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest,chi2

In [4]:
df=pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
df.drop(columns=["PassengerId","Name","Ticket","Cabin"],inplace=True)


In [6]:
df.isnull().sum()

Survived     0
Pclass       0
Sex          0
Age         86
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64

In [7]:
X_train,X_test,Y_train,Y_test=train_test_split(df.drop(columns="Survived"),df['Survived'],test_size=0.2,random_state=42)
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
336,2,male,32.0,0,0,13.0000,S
31,2,male,24.0,2,0,31.5000,S
84,2,male,NaN,0,0,10.7083,Q
287,1,male,24.0,1,0,82.2667,S
317,2,male,19.0,0,0,10.5000,S
...,...,...,...,...,...,...,...
71,3,male,21.0,0,0,7.8958,S
106,3,male,21.0,0,0,7.8208,Q
270,1,male,46.0,0,0,75.2417,C
348,2,male,24.0,0,0,13.5000,S


In [8]:
trf1=ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),
    ('Fare',SimpleImputer(),[5])
],remainder='passthrough')

In [9]:
trf2=ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown="ignore"),[1,6])
],remainder='passthrough')

In [10]:
trf3=ColumnTransformer([
    ('scale',MinMaxScaler(),slice(1,10))
])

In [11]:
trf4=SelectKBest(score_func=chi2,k=7)

In [12]:
trf5=DecisionTreeClassifier()

In [13]:
Pipe=Pipeline([('trf1',trf1),('trf2',trf2),('trf3',trf3),('trf4',trf4),('trf5',trf5)])
# make_pipeline(trf1,trf2,trf3,trf4,trf5)   

In [14]:
Pipe.fit(X_train,Y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('Fare', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
y_pred=Pipe.predict(X_test)

In [16]:
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [18]:
from sklearn.metrics import accuracy_score,confusion_matrix
# accuracy_score(Y_test,y_pred)

confusion_matrix(Y_test,y_pred)

array([[50,  0],
       [34,  0]])

In [19]:
import pickle
pickle.dump(Pipe,open('model/pipe.pkl','wb'))